In [1]:
!pip install -U transformers datasets peft trl accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 40.5 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0


In [10]:
pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 61.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [2]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [7]:
from datasets import load_dataset

dataset = load_dataset(
    "trl-lib/Capybara",
    split="train"
)

print(dataset)

README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 37.2MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  482kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15806 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset({
    features: ['source', 'messages', 'num_turns'],
    num_rows: 15806
})


In [8]:
print(dataset[0])

{'source': 'GPT4LLM', 'messages': [{'content': 'Recommend a movie to watch.\n', 'role': 'user'}, {'content': 'I would recommend the movie, "The Shawshank Redemption" which is a classic drama film starring Tim Robbins and Morgan Freeman. This film tells a powerful story about hope and resilience, as it follows the story of a young man who is wrongfully convicted of murder and sent to prison. Amidst the harsh realities of prison life, the protagonist forms a bond with a fellow inmate, and together they navigate the challenges of incarceration, while holding on to the hope of eventual freedom. This timeless movie is a must-watch for its moving performances, uplifting message, and unforgettable storytelling.', 'role': 'assistant'}, {'content': 'Describe the character development of Tim Robbins\' character in "The Shawshank Redemption".', 'role': 'user'}, {'content': 'In "The Shawshank Redemption", Tim Robbins plays the character of Andy Dufresne, a banker who is wrongfully convicted of mur

In [12]:
small_dataset = dataset.select(range(500))

print(len(small_dataset))
small_dataset

500


Dataset({
    features: ['source', 'messages', 'num_turns'],
    num_rows: 500
})

In [13]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    task_type="CAUSAL_LM"
)

In [1]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [14]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./qwen-lora",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

trainer = SFTTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    args=training_args,
    train_dataset=small_dataset,
    peft_config=peft_config
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

In [15]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
10,1.675442
20,1.603690
30,1.451252
40,1.368005
50,1.421405
60,1.358767


TrainOutput(global_step=63, training_loss=1.4843167047651986, metrics={'train_runtime': 836.1603, 'train_samples_per_second': 0.598, 'train_steps_per_second': 0.075, 'total_flos': 984924753838080.0, 'train_loss': 1.4843167047651986, 'entropy': 1.6255569815635682, 'num_tokens': 366653.0, 'mean_token_accuracy': 0.6139251589775085, 'epoch': 1.0})

In [16]:
trainer.save_model("./qwen-lora")

In [17]:
import os

output_dir = "./qwen-lora"

print("Files in trained model folder:")

for root, dirs, files in os.walk(output_dir):
    for file in files:
        print(os.path.join(root, file))

Files in trained model folder:
./qwen-lora/README.md
./qwen-lora/chat_template.jinja
./qwen-lora/training_args.bin
./qwen-lora/adapter_config.json
./qwen-lora/adapter_model.safetensors
./qwen-lora/tokenizer_config.json
./qwen-lora/tokenizer.json
./qwen-lora/checkpoint-63/README.md
./qwen-lora/checkpoint-63/optimizer.pt
./qwen-lora/checkpoint-63/chat_template.jinja
./qwen-lora/checkpoint-63/training_args.bin
./qwen-lora/checkpoint-63/scheduler.pt
./qwen-lora/checkpoint-63/rng_state.pth
./qwen-lora/checkpoint-63/adapter_config.json
./qwen-lora/checkpoint-63/trainer_state.json
./qwen-lora/checkpoint-63/adapter_model.safetensors
./qwen-lora/checkpoint-63/tokenizer_config.json
./qwen-lora/checkpoint-63/tokenizer.json


In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_path = "./qwen-lora"

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16
).to("cuda")

model = PeftModel.from_pretrained(
    base_model,
    adapter_path
).to("cuda")

model.eval()

print("Base model + LoRA adapter loaded successfully!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Base model + LoRA adapter loaded successfully!


In [19]:
def ask_model(question):
    prompt = f"""### Instruction:
{question}

### Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True
        )

    response = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    print(response)

In [20]:
ask_model("What is artificial intelligence?")

### Instruction:
What is artificial intelligence?

### Response:
Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as perception, learning, reasoning, and decision-making. AI involves the use of algorithms and machine learning techniques to enable computers to learn from data and make decisions based on that data.

Some key features of AI include:

1. Perception: The ability to process information about its environment and recognize patterns.
2. Learning: The ability to improve its performance over time through experience and trial and


In [21]:
ask_model("Explain the topic that was taught in the training examples.")

### Instruction:
Explain the topic that was taught in the training examples.

### Response:
The instruction asks to explain a specific topic from the training examples. However, there is no mention of any particular training example or topic. Therefore, I cannot provide an explanation for the topic without additional context. 

However, if you can specify which training examples were used and what the topics were, I would be happy to try to provide a response based on those details. Alternatively, if you have a specific topic in mind, please let me know, and I will do my best to provide an explanation


In [22]:
def ask_base_model(question):
    prompt = f"""### Instruction:
{question}

### Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        output = base_model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True
        )

    response = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    print(response)

In [23]:
ask_base_model("What is AI")

### Instruction:
What is AI

### Response:
AI stands for Artificial Intelligence. It refers to the simulation of human intelligence in machines that are programmed to think and act like humans. AI involves developing algorithms, data, and machine learning techniques to enable computers and other electronic devices to perform tasks that typically require human intelligence, such as recognizing patterns, making decisions, understanding language, and solving complex problems. The field of AI has seen significant advancements over the years, with the development of deep learning, natural language processing, computer vision, robotics, and more.


In [24]:
ask_base_model("If John has 5 apples, gives 2 to Mary, gets 4 more from an apple tree, and then drops half of what he has on his way home, how many apples does he arrive with? Break it down step by step")

### Instruction:
If John has 5 apples, gives 2 to Mary, gets 4 more from an apple tree, and then drops half of what he has on his way home, how many apples does he arrive with? Break it down step by step

### Response:
1. John starts with 5 apples.
2. He gives 2 apples to Mary, so now he has \(5 - 2 = 3\) apples left.
3. He receives 4 more apples from an apple tree, so now he has \(3 + 4 = 7\) apples.
4. He drops half of the remaining apples, which is \(7 / 2 = 3.5\). Since he can't drop half an apple, we'll round this


In [25]:
ask_model("If John has 5 apples, gives 2 to Mary, gets 4 more from an apple tree, and then drops half of what he has on his way home, how many apples does he arrive with? Break it down step by step")

### Instruction:
If John has 5 apples, gives 2 to Mary, gets 4 more from an apple tree, and then drops half of what he has on his way home, how many apples does he arrive with? Break it down step by step

### Response:
1. **Initial number of apples**: John starts with 5 apples.
2. **Giving 2 apples to Mary**: He gives away 2 apples to Mary.
   \[
   5 - 2 = 3
   \]
   Now John has 3 apples left.
3. **Getting 4 more apples from an apple tree**: He receives 4 more apples.
   \[
   3 + 4 = 7
   \]
   Now John has 


In [26]:
ask_base_model("Can you recommend three popular sci-fi books?")

### Instruction:
Can you recommend three popular sci-fi books?

### Response:
Certainly! Here are three highly recommended sci-fi novels:

1. **"The Martian" by Andy Weir**: This novel tells the story of astronaut Mark Watney, who is stranded on Mars after his spacecraft crashes and he must rely on his own ingenuity to survive.

2. **"The Hitchhiker's Guide to the Galaxy" by Douglas Adams**: Published in 1979, this book follows the adventures of Arthur Dent, a space traveler, as he travels through the universe


In [27]:
ask_model("Can you recommend three popular sci-fi books?")

### Instruction:
Can you recommend three popular sci-fi books?

### Response:
Sure, here are three popular sci-fi books:

1. "The Left Hand of Darkness" by Ursula K. Le Guin - This book is a classic science fiction novel that explores the concept of gender identity and the impact it has on society.

2. "The Martian" by Andy Weir - This is a science fiction thriller that follows an astronaut who must navigate a dangerous mission to Mars to find a way back home.

3. "The Hitchhiker's Guide to the Galaxy"


In [30]:
ask_base_model("Recommend a movie to watch?")

### Instruction:
Recommend a movie to watch?

### Response:
One highly recommended movie is "The Grand Budapest Hotel" directed by Wes Anderson. It's a charming, whimsical adventure set in 1940s Budapest that explores the lives of several characters and their interactions with each other and with the hotel itself. The film has won numerous awards and is considered one of the best films of all time. It's a must-see for anyone who enjoys quirky, surreal, and whimsical stories. Additionally, it offers a glimpse into the life of a


In [31]:
ask_model("Recommend a movie to watch")

### Instruction:
Recommend a movie to watch

### Response:
One highly recommended movie is "The Shawshank Redemption" directed by Frank Darabont. The movie tells the story of Andy Dufresne, who is wrongly convicted of murder and sentenced to life in prison. Despite his difficult circumstances, he remains optimistic and determined to find hope in his new world. As he navigates the challenges of being a prisoner, he forms strong bonds with fellow inmates and builds a sense of community. The film explores themes such as friendship, perseverance, and the power


In [28]:
dataset = load_dataset("trl-lib/Capybara", split="train")
small_dataset = dataset.select(range(500))

In [29]:
print(small_dataset[0])

{'source': 'GPT4LLM', 'messages': [{'content': 'Recommend a movie to watch.\n', 'role': 'user'}, {'content': 'I would recommend the movie, "The Shawshank Redemption" which is a classic drama film starring Tim Robbins and Morgan Freeman. This film tells a powerful story about hope and resilience, as it follows the story of a young man who is wrongfully convicted of murder and sent to prison. Amidst the harsh realities of prison life, the protagonist forms a bond with a fellow inmate, and together they navigate the challenges of incarceration, while holding on to the hope of eventual freedom. This timeless movie is a must-watch for its moving performances, uplifting message, and unforgettable storytelling.', 'role': 'assistant'}, {'content': 'Describe the character development of Tim Robbins\' character in "The Shawshank Redemption".', 'role': 'user'}, {'content': 'In "The Shawshank Redemption", Tim Robbins plays the character of Andy Dufresne, a banker who is wrongfully convicted of mur